# 08 — Business Impact

Statistical results need to be translated into business decisions.
A p-value alone cannot tell a product manager what to do.

This notebook covers:

1. **Revenue impact of the observed lift** — point estimate
2. **Confidence interval on revenue** — range of plausible outcomes
3. **Expected value framework** — weighing uncertain impact against shipping cost
4. **Experiment ROI** — was running this test worth it?
5. **Decision narrative** — what to actually say in the readout
6. **Experiment write-up template** — repeatable format for every test

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import two_proportion_ztest

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

ALPHA = 0.05

df = pd.read_csv("../data/ab_data_clean.csv")

# Derive revenue per converter from real data (purchase_amount > 0 for converted users)
if "purchase_amount" in df.columns:
    REVENUE_PER_CONVERT = df[df["converted"]==1]["purchase_amount"].mean()
    print(f"Average revenue per conversion (from data): ${REVENUE_PER_CONVERT:.2f}")
else:
    REVENUE_PER_CONVERT = 40.0
    print(f"purchase_amount not found — using default: ${REVENUE_PER_CONVERT:.2f}")

WEEKLY_VISITORS = 50_000  # new visitors per week (adjust to your traffic)
SHIP_COST       = 5_000   # estimated one-time engineering cost to ship ($)

grp = df.groupby("group")["converted"].agg(conversions="sum", users="count")
grp["rate"] = grp["conversions"] / grp["users"]
ctrl, trt = grp.loc["control"], grp.loc["treatment"]

z, p_val, p_c, p_t, lift_abs, lift_rel = two_proportion_ztest(
    ctrl.conversions, ctrl.users, trt.conversions, trt.users
)

print(f"Control rate:   {p_c:.4%}")
print(f"Treatment rate: {p_t:.4%}")
print(f"Absolute lift:  {lift_abs:+.4%}")
print(f"p-value:        {p_val:.4f}")

## 1. Revenue Impact — Point Estimate

We use the **actual mean purchase amount** from the dataset as our revenue-per-conversion.
This is more accurate than a hypothetical figure and shows how to connect experiment
results to real financial outcomes.

In [ ]:
# Point estimate: weekly incremental revenue
incremental_weekly_conversions = lift_abs * WEEKLY_VISITORS
incremental_weekly_revenue     = incremental_weekly_conversions * REVENUE_PER_CONVERT
incremental_annual_revenue     = incremental_weekly_revenue * 52

print("=" * 50)
print("  Revenue Impact — Point Estimate")
print("=" * 50)
print(f"  Observed lift            : {lift_abs:+.4%}")
print(f"  Weekly visitors          : {WEEKLY_VISITORS:,}")
print(f"  Incremental conversions/week: {incremental_weekly_conversions:+,.0f}")
print(f"  Revenue per conversion   : ${REVENUE_PER_CONVERT:,}")
print(f"  Incremental revenue/week : ${incremental_weekly_revenue:+,.0f}")
print(f"  Incremental revenue/year : ${incremental_annual_revenue:+,.0f}")
print("=" * 50)

## 2. Confidence Interval on Revenue

Point estimates are misleading on their own. The CI on the lift translates directly
into a CI on annual revenue — giving decision-makers a range, not a false precision.

In [ ]:
se_diff = np.sqrt(p_c*(1-p_c)/ctrl.users + p_t*(1-p_t)/trt.users)
z_crit  = stats.norm.ppf(1 - ALPHA/2)
diff_lo = lift_abs - z_crit * se_diff
diff_hi = lift_abs + z_crit * se_diff

rev_lo = diff_lo * WEEKLY_VISITORS * REVENUE_PER_CONVERT * 52
rev_hi = diff_hi * WEEKLY_VISITORS * REVENUE_PER_CONVERT * 52

print(f"95% CI on lift:           [{diff_lo:+.4%}, {diff_hi:+.4%}]")
print(f"95% CI on annual revenue: [${rev_lo:+,.0f}, ${rev_hi:+,.0f}]")

# Visualize revenue distribution
lift_samples = np.random.normal(lift_abs, se_diff, 50_000)
rev_samples  = lift_samples * WEEKLY_VISITORS * REVENUE_PER_CONVERT * 52

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rev_samples / 1000, bins=80, color="steelblue", alpha=0.7, density=True)
ax.axvline(0, color="gray", linewidth=1.5, linestyle="--", label="No impact")
ax.axvline(incremental_annual_revenue/1000, color="darkgreen", linewidth=2,
           label=f"Point estimate: ${incremental_annual_revenue/1000:+.0f}k")
ax.axvline(rev_lo/1000, color="tomato", linewidth=1.5, linestyle=":",
           label=f"95% CI lower: ${rev_lo/1000:+.0f}k")
ax.axvline(rev_hi/1000, color="tomato", linewidth=1.5, linestyle=":",
           label=f"95% CI upper: ${rev_hi/1000:+.0f}k")
ax.set_xlabel("Annual incremental revenue ($k)")
ax.set_ylabel("Density")
ax.set_title("Distribution of plausible annual revenue impacts")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Expected Value Framework

When the result is uncertain (p > 0.05, CI spans zero), expected value helps frame
the decision by weighing the probability of a positive effect against its magnitude.

$$EV[\text{ship}] = P(\text{positive}) \times \text{upside} + P(\text{negative}) \times \text{downside} - \text{shipping cost}$$

If EV[ship] > 0 → lean toward shipping.
If EV[ship] < 0 → lean toward not shipping or running a larger experiment.

In [ ]:
# Proportion of the revenue distribution that is positive
p_positive = np.mean(rev_samples > 0)
p_negative = 1 - p_positive

mean_upside   = rev_samples[rev_samples > 0].mean() if p_positive > 0 else 0
mean_downside = rev_samples[rev_samples < 0].mean() if p_negative > 0 else 0

ev_ship = p_positive * mean_upside + p_negative * mean_downside - SHIP_COST

print(f"P(positive annual revenue) : {p_positive:.1%}")
print(f"E[revenue | positive]      : ${mean_upside:,.0f}")
print(f"P(negative annual revenue) : {p_negative:.1%}")
print(f"E[revenue | negative]      : ${mean_downside:,.0f}")
print(f"Shipping cost              : ${SHIP_COST:,}")
print()
print(f"E[value of shipping]       : ${ev_ship:,.0f}")
print()
if ev_ship > 0:
    print("Expected value is POSITIVE — lean toward shipping despite uncertainty.")
else:
    print("Expected value is NEGATIVE — lean toward NOT shipping, or run a larger test.")

## 4. Experiment ROI

Running the experiment itself has a cost — engineering time, traffic allocation
(some users got the worse experience), and analyst time.

The **value of information** from the experiment is the difference between the
best decision we can make *with* the data vs. *without* it.

In [ ]:
experiment_cost = 15_000   # est: eng time + analyst time ($)

# Without the experiment: we either ship blindly or do nothing
# With the experiment: we have evidence to inform the decision

# Cost of a wrong decision (shipping a page that hurts): magnitude of downside
cost_of_wrong_ship = abs(mean_downside) if p_negative > 0 else 0

# Value of knowing before shipping = expected cost avoided by not shipping a bad change
value_of_info = p_negative * cost_of_wrong_ship

print(f"Experiment cost        : ${experiment_cost:>10,.0f}")
print(f"Value of information   : ${value_of_info:>10,.0f}")
print(f"Net experiment ROI     : ${value_of_info - experiment_cost:>+10,.0f}")
print()
print("Note: this is a simplified VOI calculation.")
print("Full Bayesian VOI analysis accounts for prior beliefs about effect sizes.")

## 5. The Decision Narrative

Every experiment should end with a clear, written decision — not just numbers.
Here is the narrative for our experiment.

---

### Experiment Readout: Landing Page Redesign

**Result:** The new landing page did not produce a statistically significant change
in conversion rate (p = 0.19, 95% CI on lift: [−0.5%, +0.1%]).

**What this means:** We cannot conclude the new page improves or worsens conversion.
The observed −0.2pp difference is within the range expected from random variation.

**What this does NOT mean:** The page has no effect. We may lack the statistical power
to detect effects smaller than ±0.3pp with our current traffic.

**Recommendation:** Do not ship the redesign based on this experiment.
Options:
1. Run a larger experiment (targeting 2pp MDE with 80% power requires ~7,600 users per group)
2. Redesign the treatment to make a more substantial change before re-testing
3. Investigate whether specific segments (mobile, new users) respond differently

---

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 1. Conversion rates
ax = axes[0]
ax.bar(["Control", "Treatment"], [p_c*100, p_t*100], color=["steelblue","coral"], width=0.5)
ax.set_ylabel("Conversion rate (%)")
ax.set_title("Conversion rates")
ax.set_ylim(10, 14)
for i, (r, label) in enumerate([(p_c, "Control"),(p_t, "Treatment")]):
    ax.text(i, r*100 + 0.05, f"{r:.2%}", ha="center", fontweight="bold")

# 2. CI on lift
ax = axes[1]
ax.axvline(0, color="gray", linewidth=1.5, linestyle="--")
ax.errorbar(lift_abs*100, 0.5,
            xerr=[[abs(diff_lo-lift_abs)*100],[abs(diff_hi-lift_abs)*100]],
            fmt="o", color="steelblue", capsize=10, markersize=9, linewidth=2.5)
ax.set_yticks([])
ax.set_xlabel("Lift (pp)")
ax.set_title("95% CI on lift")

# 3. Revenue distribution
ax = axes[2]
ax.hist(rev_samples/1000, bins=60, color="steelblue", alpha=0.6, density=True)
ax.axvline(0, color="tomato", linewidth=1.5, linestyle="--")
ax.set_xlabel("Annual revenue impact ($k)")
ax.set_title("Revenue uncertainty")

plt.suptitle("Experiment Summary: Landing Page Redesign", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Experiment Write-Up Template

Use this structure for every experiment readout:

```
## Experiment: [Name]

**Hypothesis:** [What we expected and why]
**Treatment:** [What changed]
**Primary metric:** [Metric name and formula]
**Date range:** [Start] – [End]  |  **N:** [Control] / [Treatment]

### Results
| Metric | Control | Treatment | Lift | p-value | Significant? |
|--------|---------|-----------|------|---------|--------------|
| [primary] | | | | | |
| [guardrail 1] | | | | | |

### Decision
[Ship / Do not ship / Inconclusive — retest]

### Why
[2–3 sentences translating the stats into business language]

### Next steps
[Concrete follow-up actions]
```

---

## End of Portfolio

You have now seen the full A/B testing lifecycle:

| Notebook | Concept |
|---|---|
| 01 | Experiment design — hypothesis, MDE, sample size |
| 02 | Data validation — SRM, deduplication, mismatches |
| 03 | Statistical analysis — z-test, p-values, CIs |
| 04 | Practical significance — effect size, decision quadrants |
| 05 | Segment analysis — Simpson's Paradox, interaction effects |
| 06 | Multiple testing — Bonferroni, BH-FDR |
| 07 | Business impact — revenue CIs, expected value, decision narrative |

The full code is reusable via `src/ab_stats.py`.